# Topic: Experiment Tracking (MLflow)

## Definition (30-second explanation)
*   Think of experiment tracking as a highly detailed financial ledger, but for machine learning models. Instead of money, it records exactly what ingredients (data, hyperparameters) went into a model and what the outcome (accuracy, loss) was.
*   It ensures you never face the nightmare of training a fantastic model but forgetting the exact settings used to create it.

## Why Interviewers Ask This
*   Anyone can train a model in a messy Jupyter notebook, but interviewers need proof you can build reproducible, production-ready systems. 
*   In regulated industries, auditors require concrete proof that a production model was built with specific data and parameters.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:** Without tracking, it becomes impossible to reproduce previous results, understand why one model outperforms another, or collaborate effectively with a team.
*   **The Mechanism:** Tools like MLflow act as a wrapper around your training script. During a run, they intercept and systematically log everything to a central database: parameters (learning rate), metrics (AUC), environment states (library versions), and artifacts (saved model files).
*   **The Trade-off:** Implementing tracking adds slight code overhead and requires maintaining a central tracking server for team collaboration, but the cost of lost experiments far outweighs this setup effort.

## When to Use
*   Universally required for every single machine learning or deep learning training script you write.
*   Crucial when conducting hyperparameter tuning (e.g., grid search) to compare dozens of configurations.

## Advantages
*   Guarantees reproducibility and enables easy comparison across experiments.
*   Critical for ML governance, regulatory compliance, and auditing.
*   Allows seamless model packaging and handoffs to MLOps deployment teams.

## Limitations
*   Local, file-based MLflow (`mlruns` folder) does not scale well for team collaboration without setting up a remote tracking server.
*   Experiment trackers like MLflow and W&B are not natively designed to version massive, terabyte-scale datasets (you need tools like DVC for that).

## Common Comparisons
*   **MLflow vs. Weights & Biases (W&B):** MLflow is the open-source standard with a strong local/cloud model registry, great for teams using multiple frameworks. W&B offers a richer UI with real-time charts and sweep automation, heavily favored in deep learning research.
*   **MLflow vs. DVC:** DVC is heavily focused on data versioning when tracking massive datasets is critical, whereas MLflow focuses on logging the training run parameters and metrics.

## Common Interview Traps
*   **Assuming Tracking = Reproducibility:** Just logging parameters isn't enough. If you forget to set your random seeds (`np.random.seed`, `torch.manual_seed`), the experiment remains non-reproducible.
*   **Ignoring the Environment:** Forgetting to log the Git commit hash, Python version, and library requirements. A logged model is useless if you don't know the exact dependency versions needed to load it.

## Python Syntax
```python
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Set the experiment name to group related runs logically
mlflow.set_experiment('breast_cancer_classification')

# Define configurations to test
cfg = {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}

# 2. Start the MLflow tracking context manager
with mlflow.start_run()
    
    # 3. Log hyperparameters BEFORE training
    mlflow.log_params(cfg)
    
    # Train the model
    model = RandomForestClassifier(**cfg)
    model.fit(X_train, y_train)
    
    # Generate predictions and calculate metrics
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    # 4. Log evaluation metrics
    mlflow.log_metrics({'accuracy': acc})
    
    # 5. Log the actual trained model object as a saved artifact
    mlflow.sklearn.log_model(model, 'random_forest_model')
    
print('All experiments logged. Run: mlflow ui to see results.')
```

## 45-Second Interview Answer
"Experiment tracking systematically logs parameters, metrics, environment details, and artifacts for every training run. Without it, collaboration and auditing are nearly impossible. Using tools like MLflow, I ensure complete reproducibility by locking down random seeds and tracking everything from the Git commit hash to the final model weights. This transforms messy scripts into a governed, production-ready pipeline."

## Practice Questions:

### Q1:
**Question:**
How do you ensure reproducibility of a machine learning experiment?

**Answer:**
"Reproducibility requires strict control over all sources of randomness and versioning. First, I set fixed random seeds across all libraries, such as `numpy`, `sklearn`, and `torch`. Second, I enforce deterministic algorithms, avoiding non-deterministic CUDA operations where possible. From an infrastructure standpoint, I pin all library versions in a `requirements.txt` or `conda` environment, use tools like DVC to version the exact dataset hash, and log the Git commit hash of the code. Finally, I use an experiment tracker like MLflow or W&B to systematically log every hyperparameter and metric for the run."

**Common Mistakes Candidates Make:**
*   Failing to explicitly mention setting random seeds (`np.random.seed(42)`, `tf.random.set_seed(42)`); this is the single most common reason tracked experiments still yield varying results.

**Likely Interviewer Follow-up:**
"If you are training on a dataset that updates daily, how do you ensure the MLflow tracking perfectly maps to the exact data state used on a specific day?

(Answer:
"MLflow tracks parameters well but isn't built for raw data storage. To ensure perfect reproducibility with dynamic data, I would link the MLflow run to an immutable data snapshot by versioning the data and logging the dataset hash with the experiment. Specifically, I would use one of three methods:
1. **Data Version Control (DVC):** Track the dataset with DVC and log the generated hash as an MLflow parameter.
2. **Data Lake Time Travel:** If using modern data architecture like Delta Lake or Apache Iceberg, log the exact table version or timestamp in MLflow to query the exact historical state later.
3. **Cryptographic Hashing:** For standard files, compute the MD5 or SHA-256 hash of the dataset at runtime and log the hash string to MLflow. If the hash changes, we know the data drifted.")

### Q2: From Tracking to Deployment
**Question:** 
You logged 50 experiments in MLflow and found the best XGBoost model. How do you bridge the gap between this logged experiment and a production deployment?

**Answer:**
"To move a model from an experiment to production, I would use the **MLflow Model Registry**. The tracking server is for raw experiments, but the registry acts as the governed 'showroom' for deployment. I would register the best run and assign it a lifecycle stage, such as 'Staging' for testing, and eventually 'Production'. 
The main engineering advantage of this is abstraction: the backend serving team doesn't need to know the specific run IDs or file paths. They simply point their API to load `models:/delivery_delay_model/Production`. When I train a better model in the future, I just update the tag in the registry, and the backend automatically serves the new version without any code changes."

**Common Interview Traps:**
* Assuming the MLOps team will just manually download a `.pkl` file from your Jupyter environment or an S3 bucket. (The registry is required for version control and governance).
* Confusing the Tracking Server (for logging parameters/metrics) with the Model Registry (for managing production lifecycles).